# Intro to LangChain
LangChain is a popular framework that allows you to quickly build applications and pipelines of Large Language Models (LLMs). You can use it to create chatbots, RAGs, agents and much more.

In this notebook, we will focus on the simplest useful LangChain workflow: take a question, pass it through a prompt template, send it to a model, and inspect the result.

By the end, you should be able to:
- explain what the main LangChain building blocks do
- create a `PromptTemplate` with one dynamic variable
- connect the prompt and model into a chain with `|`
- run the same chain for one question or for a batch of questions

The main idea of the library is that we can create a _chain_ of different components to create more complex applications. These _chains_ (you can think of them as pipelines) can be made up of various components such as:
- **Prompt templates**: templates that generate prompts dynamically from user input or context.
- **LLMs**: Large Language Models are the core of LangChain. You can use any LLM that is compatible with the library, including hosted APIs and open-weight models such as Llama.
- **Tools**: Tools are functions that can be used by the LLM to perform specific tasks. For example, you can use a tool to search the web, or to access a database.
- **Agents**: Agents are components that can use LLMs and tools to perform specific tasks. They can be used to create chatbots, **R**etrieval **A**ugumentation **G**eneration (RAGs), etc.
- **Retrievers**: Retrievers are components that can be used to retrieve information from a database or a knowledge base. They can be used to create RAGs, or to retrieve information from a database.  
- **Memory**: Memory is a component that can be used to store information about the conversation. It can be used to create chatbots that can remember previous conversations, or to create RAGs that can remember previous queries.

## Lesson Map
The flow we are about to build is intentionally small so each piece is easy to understand.

```mermaid
flowchart LR
    A[Question text] --> B[PromptTemplate]
    B --> C[LangChain chain]
    C --> D[ChatGroq with gpt-oss-20b]
    D --> E[AIMessage response]
    E --> F[Rendered text output]
```

As you read the notebook, keep an eye on which cell is responsible for each box in the diagram.

## Using LLMs in LangChain

LangChain supports a wide range of providers and model families for LLMs, including OpenAI, Hugging Face, Groq, and open-weight models such as Llama.

In this notebook, we will use Groq as the provider and `openai/gpt-oss-20b` as the model. That keeps the example realistic while still staying compact enough for a first pass.

### Groq Integration
Groq is a provider of LLMs that offers high-performance inference capabilities. To use Groq with LangChain, you need to set up your API key in the `.env` file. Follow the steps in the README.md file to set up your environment.

In [ ]:
# Load credentials from the local .env file before creating the model client.
from dotenv import load_dotenv
from langchain_groq import ChatGroq
from langchain_core.prompts import PromptTemplate


# LangChain message content can be plain text or structured content.
# This helper normalizes both cases so our print statements stay simple.
def render_message_content(content: object) -> str:
    if isinstance(content, str):
        return content
    if isinstance(content, list):
        return "".join(part if isinstance(part, str) else str(part) for part in content)
    return str(content)

#### Load Credentials from .env file

In [ ]:
load_dotenv()

#### Defining the LLM (Using Groq)

We can define the LLM using the [`ChatGroq`](https://python.langchain.com/docs/integrations/chat/groq/) class from the `langchain_groq` module. 

This class allows us to specify:
+ the model - below we use `openai/gpt-oss-20b`
+ the temperature - we set it to `0.1` for more deterministic responses
+ the maximum tokens - we set it to `512` to limit the response length

Those three parameters are enough for a beginner-friendly example:
- `model` chooses the underlying LLM
- `temperature` controls how varied or deterministic the output should be
- `max_tokens` prevents very long answers when we only need a short explanation

In [ ]:
llm = ChatGroq(
    model="openai/gpt-oss-20b",
    temperature=0.1,
    max_tokens=512,
)

#### Build prompt template
A prompt is a set of instructions or input provided by a user to an LLM to guide its response. It helps the model understand the context and generate relevant output. In LangChain, we can create a prompt template using the `PromptTemplate` class.

The template below has one placeholder, `{question}`. LangChain will replace that placeholder with the value we pass in at runtime.

In [ ]:
# Keep the template minimal so it is easy to see where the dynamic variable goes.
template = """Question: {question}

Answer: """
prompt = PromptTemplate(template=template, input_variables=["question"])

The `input_variables` are defined in the template using curly braces like `{question}`. This is what turns a static string into a reusable prompt pattern.

#### Preview the Prompt Before Sending It
A useful debugging habit is to inspect the fully formatted prompt before you call the model. This helps you verify that your placeholders were replaced correctly and that the final prompt still reads naturally.

In [ ]:
# format() returns the final string that will be passed into the chain.
preview_prompt = prompt.format(question="What is a confusion matrix?")
print(preview_prompt)

#### Define Chain
A chain is a sequence of components that are executed in order to produce a final output. In LangChain, we can use the pipe symbol `|` to define a chain of components. The output of one component is passed as input to the next component in the chain.

```mermaid
flowchart LR
    A[Question text] --> B[PromptTemplate]
    B --> C[Formatted prompt]
    C --> D[ChatGroq]
    D --> E[AIMessage answer]
```

This is one of the most important LangChain ideas: instead of manually formatting prompts and then calling the model yourself every time, you can compose those steps into a reusable pipeline.

In [ ]:
# Pipe the formatted prompt directly into the model.
chain = prompt | llm

#### Invoke the Chain
We will keep the input, invocation, and output display in separate cells so you can inspect each stage.

In [ ]:
# Start with one simple question so the chain behavior is easy to inspect.
question = "What is the backpropagation algorithm?"

In [ ]:
# The dictionary key must match the template variable name: question.
answer = chain.invoke(input={"question": question})

In [ ]:
# Render the returned AIMessage content as plain text for display.
print(render_message_content(answer.content).strip())

#### Inspect the Response Object
Even though we usually print only the answer text, the chain actually returns an `AIMessage` object. Looking at the object type helps explain why we sometimes need a helper such as `render_message_content()` when working with chat-model outputs.

In [ ]:
# The chain returns a richer object than plain text.
print(type(answer).__name__)
print(answer)

#### Try a Different Prompt Style
One of the easiest ways to change model behavior is to adjust the prompt instructions while keeping the same model. Here we ask for a shorter, more structured teaching style so you can see how the wording of the prompt influences the wording of the answer.

In [ ]:
# A slightly more specific prompt often produces more controlled output.
teaching_template = """You are a patient machine-learning teaching assistant.
Question: {question}
Answer in exactly 3 short bullet points.
"""

teaching_prompt = PromptTemplate(
    template=teaching_template,
    input_variables=["question"],
)

teaching_chain = teaching_prompt | llm
teaching_answer = teaching_chain.invoke(
    {"question": "Why is gradient descent useful in machine learning?"}
)
print(render_message_content(teaching_answer.content).strip())

If we want to ask multiple questions, we can pass a list of dictionaries to `batch()`. Each dictionary follows the same schema as `invoke()`, which means each one must provide a value for the template variable `question`.

This is useful when you want to run the same prompt structure over a small dataset of related questions.

In [ ]:
# Every item in the batch is one set of prompt-template inputs.
qs = [
    {"question": "What is the backpropagation algorithm?"},
    {"question": "What is the purpose of the activation function in a neural network?"},
    {
        "question": "What is the difference between supervised and unsupervised learning?"
    },
    {"question": "Explain the concept of overfitting in machine learning."},
]

In [ ]:
# batch() applies the same chain to every item in the list.
answers = chain.batch(qs)

In [ ]:
# Zip each input question with its matching model answer for readable output.
for question, answer in zip(qs, answers):
    print("=" * 100)
    print(f"Question: {question['question']}")
    print(f"Answer: {render_message_content(answer.content).strip()}")
    print("=" * 100)

## Things to Test Next
Now that you have a basic LangChain pipeline working, here are some useful follow-up experiments:
- change the wording of the prompt and observe how the tone of the answer changes
- try a higher `temperature` and compare the consistency of repeated answers
- add more input variables, such as `audience` or `answer_length`, to the prompt template
- replace the single-question chain with a prompt that asks for JSON or bullet-point output
- compare `invoke()` and `batch()` when you want one answer versus many answers